## 0: import data
The `content_libraries/amazon_2023` folder holds a single catalog (titles + credits), so the analysis is done on one dataset instead of a Netflix vs Amazon comparison.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

titles = pd.read_csv("content_libraries/amazon_2023/amazon_titles_2023.csv")
credits = pd.read_csv("content_libraries/amazon_2023/amazon_credits_2023.csv")

## 1.0: Info on the 2023 titles dataset

In [ ]:
titles.head()
titles.info()
titles.describe()

In [ ]:
# how many null values?
print(titles.shape)
print(titles.columns)
print(titles.isna().sum())

missing_titles = titles.isnull().mean().sort_values(ascending=False)

missing_titles.plot(kind="bar", figsize=(9,3))
plt.ylabel("Proportion missing")
plt.title("Missing data in 2023 titles dataset")
plt.show()

## 1.1: Info on the 2023 credits dataset

In [ ]:
credits.head()
credits.info()
credits.describe()

In [ ]:
# how many null values?
print(credits.shape)
print(credits.columns)
print(credits.isna().sum())

missing_credits = credits.isnull().mean().sort_values(ascending=False)

missing_credits.plot(kind="bar", figsize=(9,3))
plt.ylabel("Proportion missing")
plt.title("Missing data in 2023 credits dataset")
plt.show()

## 2.0: Prepare the dataset

In [ ]:
catalog = titles.copy()
catalog["service"] = "Amazon Prime 2023"

## 3.0: Distribution of movies and TV shows

In [ ]:
sns.countplot(data=catalog, x="service", hue="type")
plt.title("Movies and TV shows in the 2023 catalog")
plt.show()

## 3.1 Percentage of movies and series

In [ ]:
pd.crosstab(
    catalog["service"],
    catalog["type"],
    normalize="index"
) * 100

## 4.0 Age of the content available

In [ ]:
catalog["content_age"] = 2023 - catalog["release_year"]

sns.boxplot(data=catalog, x="service", y="content_age")
plt.title("Age of streaming content in 2023")
plt.show()

# average release year
catalog.groupby("service")["release_year"].mean()

## 5.0 Content quality based on imdb rating

In [ ]:
sns.histplot(
    data=catalog,
    x="imdb_score",
    hue="service",
    kde=True,
    bins=20
)

plt.title("IMDb ratings of the 2023 catalog")
plt.show()


catalog.groupby("service")["imdb_score"].agg(
    ["mean", "median", "std", "count"]
)

# not sufficient: if catalog increases but the number of high-rated titles
# stays the same, it should not be punished with a lower average rating

## 5.1 Create upper bound for high-quality content measure

In [ ]:
# Define what high_quality is in this case
catalog["high_quality"] = (
    (catalog["imdb_score"] >= 7) &
    (catalog["imdb_votes"] >= 7000)
)

high_quality_stats = (
    catalog.groupby("service")
    .agg(
        total_titles=("imdb_id", "count"),
        high_quality_titles=("high_quality", "sum"),
        high_quality_share=("high_quality", "mean")
    )
)

high_quality_stats["high_quality_share"] *= 100

high_quality_stats.round(2)

# - high_quality_share:
#   What proportion of the catalog is highly rated?
# - high_quality_titles:
#   How many highly rated titles can consumers access?

In [ ]:
# Bayesian/weighted rating
# - we give more weight to imdb scores that have more reviews

# Overall average IMDb score
C = catalog["imdb_score"].mean()

# prior strength: the title's own IMDb score and the overall average each get roughly 50% weight (needs defined)
m = 3000

# Bayesian weighted IMDb score
catalog["weighted_imdb"] = (
    (catalog["imdb_votes"] / (catalog["imdb_votes"] + m))
    * catalog["imdb_score"]
    +
    (m / (catalog["imdb_votes"] + m))
    * C
)

catalog.groupby("service")["weighted_imdb"].mean().round(2)

In [ ]:
# answers: What is the overall quality of the catalog when we account for the reliability of IMDb ratings?
quality_comparison = (
    catalog.groupby("service")
    .agg(
        number_of_titles=("imdb_id", "count"),
        average_imdb=("imdb_score", "mean"),
        average_weighted_imdb=("weighted_imdb", "mean"),
        median_imdb_votes=("imdb_votes", "median"),
        high_quality_titles=("high_quality", "sum"),
        high_quality_share=("high_quality", "mean")
    )
)

quality_comparison["high_quality_share"] *= 100

quality_comparison.round(2)

## 6.0 Genre distribution

In [ ]:
catalog["genres"] = catalog["genres"].fillna("[]")
catalog["genres"] = catalog["genres"].apply(ast.literal_eval)

genres = catalog.explode("genres")

genres["genres"] = genres["genres"].str.strip()

# number of movies and tv shows in each genre
genre_counts = genres.groupby("genres").size().sort_values(ascending=False)

print(genre_counts)

In [ ]:
# number of tv shows and movies across genres
# note: a movie or a show can have multiple genres
genre_counts = genres.groupby(["genres", "service"]).size().unstack(fill_value=0)

genre_counts.sort_values("Amazon Prime 2023", ascending=False).head(10).plot(kind="bar", figsize=(12, 6))

plt.xlabel("Genre")
plt.ylabel("Number of titles")
plt.title("Top 10 Genres in the 2023 catalog")
plt.xticks(rotation=45)
plt.legend(title="Service")
plt.tight_layout()
plt.show()

## 6.1 Which genre provide the highest-quality content?

In [ ]:
# based on imdb_score
genre_ratings = (
    genres.groupby(["genres", "service"])["imdb_score"]
    .mean()
    .unstack()
)

genre_ratings

In [ ]:
# based on the bayesian weighted score
genre_ratings_weighted = (
    genres.groupby(["genres", "service"])["weighted_imdb"]
    .mean()
    .unstack()
    .round(2)
)
genre_ratings_weighted

## 7.0: Country of origin

## 8.0: Comparison Amazon 2022 vs Amazon 2023

In [ ]:
amazon_2022 = pd.read_csv("content_libraries/amazon_2022/amazon_titles_2022.csv")
amazon_2022["service"] = "Amazon Prime 2022"
amazon_2022["year"] = 2022

amazon_2023 = pd.read_csv("content_libraries/amazon_2023/amazon_titles_2023.csv")
amazon_2023["service"] = "Amazon Prime 2023"
amazon_2023["year"] = 2023

years = pd.concat([amazon_2022, amazon_2023], ignore_index=True)

# titles added / removed between the two snapshots
ids_2022, ids_2023 = set(amazon_2022["id"]), set(amazon_2023["id"])
print("2022 titles:", len(ids_2022))
print("2023 titles:", len(ids_2023))
print("kept:", len(ids_2022 & ids_2023))
print("removed since 2022:", len(ids_2022 - ids_2023))
print("added in 2023:", len(ids_2023 - ids_2022))

## 8.1: Movies and TV shows per year

In [ ]:
sns.countplot(data=years, x="year", hue="type")
plt.title("Movies and TV shows: 2022 vs 2023")
plt.show()

pd.crosstab(years["year"], years["type"], normalize="index") * 100

## 8.2: Content age

In [ ]:
years["content_age"] = years["year"] - years["release_year"]

sns.boxplot(data=years, x="year", y="content_age")
plt.title("Age of Amazon content")
plt.show()

years.groupby("year")[["release_year", "content_age"]].mean().round(2)

## 8.3: Quality (same definitions as above)

In [ ]:
years["high_quality"] = (years["imdb_score"] >= 7) & (years["imdb_votes"] >= 7000)

# use one shared prior mean so the two years are comparable
C_all = years["imdb_score"].mean()
m = 3000
years["weighted_imdb"] = (
    (years["imdb_votes"] / (years["imdb_votes"] + m)) * years["imdb_score"]
    + (m / (years["imdb_votes"] + m)) * C_all
)

yearly_quality = (
    years.groupby("year")
    .agg(
        number_of_titles=("id", "count"),
        average_imdb=("imdb_score", "mean"),
        average_weighted_imdb=("weighted_imdb", "mean"),
        median_imdb_votes=("imdb_votes", "median"),
        high_quality_titles=("high_quality", "sum"),
        high_quality_share=("high_quality", "mean")
    )
)
yearly_quality["high_quality_share"] *= 100
yearly_quality.round(2)

## 8.4: Genres

In [ ]:
years["genres"] = years["genres"].fillna("[]").apply(ast.literal_eval)
genres_years = years.explode("genres")
genres_years["genres"] = genres_years["genres"].str.strip()

genre_year_counts = genres_years.groupby(["genres", "year"]).size().unstack(fill_value=0)
genre_year_counts["change"] = genre_year_counts[2023] - genre_year_counts[2022]
genre_year_counts["change_pct"] = (genre_year_counts["change"] / genre_year_counts[2022] * 100).round(1)

genre_year_counts.sort_values(2023, ascending=False).head(10)[[2022, 2023]].plot(kind="bar", figsize=(12, 6))
plt.xlabel("Genre")
plt.ylabel("Number of titles")
plt.title("Top 10 genres: 2022 vs 2023")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

genre_year_counts.sort_values("change", ascending=False)